# Evaluation — gemma4:e4b (Ollama)

## Experiment metadata

| | Whole-doc |
|---|---|
| **Prompt strategy** | All blocks in 1 request · model sees full document |
| **Input files** | `data/output/labeled/gemma4-e4b_whole_doc/sampleN.json` |
| **Regenerate** | `dmpbridge-experiment experiments/gemma4-e4b-wholedoc.yaml` |
| **Provider** | Ollama (local) |
| **Model size** | 4B parameters (E4B quantisation) |

**Labels (5):** `title` · `section.title` · `section.description` · `question.text` · `answer.text`  
**Evaluation:** block-level accuracy + F1 against `data/input/ground_truth/sampleN_dmp.json`  — no new files are written.

---

## Sections
1. Accuracy summary — per-sample table
2. Per-sample accuracy — bar chart
3. Confusion matrix — raw counts and row-normalised recall
4. Precision / Recall / F1 — per label
4b. Confidence score analysis
5. Mislabeled blocks — error breakdown and full detail table

In [ ]:
MODEL_NAME    = "gemma4-e4b"
MODEL_DISPLAY = "gemma4:e4b"
COLOR         = "#7c3aed"
CMAP          = "Purples"

In [ ]:
from dmpbridge.evaluation.evaluate import (
    extract_gold, evaluate_sample, match,
    LABELS, SHORT, LLM_DIR, MANUAL_DIR, NO_MATCH,
    load_method, compute_f1_rows, confusion_matrix_df,
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", palette="muted")
plt.rcParams.update({
    "figure.facecolor":"white", "axes.facecolor":"white",
    "axes.edgecolor":"#555555", "axes.linewidth":0.8,
    "grid.color":"#dddddd",    "grid.linewidth":0.5,
    "text.color":"#111111",    "axes.labelcolor":"#111111",
    "xtick.color":"#111111",   "ytick.color":"#111111",
    "font.size":11, "axes.titlesize":13, "axes.labelsize":11,
    "xtick.labelsize":10, "ytick.labelsize":10,
    "legend.fontsize":10, "legend.frameon":True,
    "figure.dpi":120,
})

df, conf, errs = load_method(f"{MODEL_NAME}_whole_doc")

if df is None or df.empty:
    print(f"No results found — run: dmpbridge-experiment experiments/gemma4-e4b-wholedoc.yaml")
else:
    tc, tn = int(df["correct"].sum()), int(df["total"].sum())
    print(f"Whole-doc ({MODEL_NAME}_whole_doc) : {tc}/{tn} ({tc/tn*100:.1f}%)")

## 1 — Accuracy summary

- **Accuracy** = blocks where the predicted label exactly matches gold.
- **Delta** = whole-doc minus batched. Positive = whole-doc is better on that sample.
- A single bad sample can pull the overall number down significantly — check the per-sample view next.

In [ ]:
def print_table(df, label):
    hdr = f"{'Sample':<12}  {'Total':>5}  {'Correct':>7}  {'Errors':>6}  {'Accuracy':>8}  Formula"
    print(f"  {label}")
    print("  " + "-" * len(hdr))
    for row in df.itertuples():
        print(f"  {row.sample:<12}  {row.total:>5}  {row.correct:>7}  "
              f"{row.errors:>6}  {row.accuracy*100:>7.1f}%  {row.formula}")
    print("  " + "-" * len(hdr))
    tn, tc = df["total"].sum(), df["correct"].sum()
    print(f"  {'TOTAL':<12}  {tn:>5}  {tc:>7}  {tn-tc:>6}  {tc/tn*100:>7.1f}%  {tc}/{tn}\n")

print_table(df, f"Whole-doc — {MODEL_NAME}")

## 2 — Per-sample accuracy

Hatched bars = whole-doc variant.  
Large gaps between the two methods on the same sample = documents where batching loses structural context.

In [ ]:
samples_ordered = df["sample"].tolist()
x = range(len(samples_ordered))

fig, ax = plt.subplots(figsize=(11, 5))
acc = df.set_index("sample").loc[samples_ordered, "accuracy"] * 100
colors = [COLOR if v >= 95 else "#f59e0b" if v >= 85 else "#dc2626" for v in acc]
bars = ax.bar(samples_ordered, acc, color=colors, width=0.6, edgecolor="white")
for bar, v in zip(bars, acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
avg = acc.mean()
ax.axhline(avg, color="#444", linestyle="--", linewidth=1.0)
ax.text(len(x)-0.45, avg+0.7, f"Mean={avg:.1f}%",
        ha="right", va="bottom", fontsize=9.5, color="#444")

ax.set_xticks(list(x)); ax.set_xticklabels(samples_ordered, rotation=30)
ax.set_ylim(30, 118); ax.set_ylabel("Block-level accuracy (%)")
ax.set_title(f"Per-sample accuracy — {MODEL_NAME} (wholedoc)", pad=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
sns.despine(); plt.tight_layout(); plt.show()

## 3 — Confusion matrix

Read **row by row**: each row is a true label, each column is what the model predicted.  
A perfect model has all mass on the diagonal.  
The **missed** column counts gold items the extractor surfaced no matching block for — a false negative from extraction, not from misclassification.  
The most common failure mode is `answer.text` being confused with `question.text` — both are researcher-written narrative and the model must distinguish them by structural position.

In [ ]:
def _row_normalize(mat):
    return mat.div(mat.sum(axis=1).replace(0, 1), axis=0)

mat   = confusion_matrix_df(conf)
mat_n = _row_normalize(mat)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
panels = [
    (axes[0], mat,   mat.values, "d", CMAP, "Raw counts  (missed = gold with no matching block)"),
    (axes[1], mat_n, mat.values, "d", CMAP, "Recall per label (row-normalised, counts annotated)"),
]

for ax, m, annot, fmt, cmap, title in panels:
    sns.heatmap(m, annot=annot, fmt=fmt, cmap=cmap, linewidths=0.6,
                linecolor="white", ax=ax, cbar=False, annot_kws={"size": 10})
    ax.set_title(title, pad=8, fontsize=11)
    ax.set_xlabel("Predicted", labelpad=6)
    ax.set_ylabel("True", labelpad=6)

plt.suptitle(
    f"Confusion matrix — {MODEL_NAME} (wholedoc)\n"
    "'missed' = gold items with no matching predicted block",
    fontsize=11, y=1.02,
)
plt.tight_layout()
plt.show()

## 4 — Precision / Recall / F1 per label

F1 balances precision and recall. Values below 80% on a label indicate a reliability risk for the pipeline.  
The delta table at the bottom shows per-label gain from whole-doc context — this is the clearest signal for deciding which strategy to use in production.

In [ ]:
df_f1 = compute_f1_rows(conf)

fig, ax = plt.subplots(figsize=(10, 5))
xi, w = range(len(LABELS)), 0.26
ax.bar([i - w for i in xi], df_f1["precision"]*100, width=w, label="Precision",
       color=COLOR, alpha=0.5, edgecolor="white")
ax.bar([i     for i in xi], df_f1["recall"]*100,    width=w, label="Recall",
       color=COLOR, alpha=0.75, edgecolor="white")
bars = ax.bar([i + w for i in xi], df_f1["f1"]*100, width=w, label="F1",
              color=COLOR, edgecolor="white")
for bar, row in zip(bars, df_f1.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f"{row.f1*100:.0f}%", ha="center", va="bottom",
            fontsize=9, fontweight="bold", color=COLOR)
ax.set_xticks(list(xi)); ax.set_xticklabels(SHORT, fontsize=10)
ax.set_ylim(0, 120); ax.set_ylabel("Score (%)")
ax.set_title(f"Whole-doc — {MODEL_NAME}", pad=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.legend(loc="upper right", framealpha=0.9)
sns.despine()

plt.suptitle(f"Precision / Recall / F1 — {MODEL_NAME}", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

display(df_f1.set_index("label").style.format("{:.1%}").background_gradient(cmap=CMAP))

## 4b — Confidence score analysis

The model reports a `confidence` field (0.0 – 1.0) alongside every label.

- **Distribution** — how often does the model express high vs low confidence per label?
- **Calibration** — when confidence is ≥ 0.9, is the model actually right ≥ 90% of the time?  
  A well-calibrated model's bar heights should track the diagonal (dashed line).
- **Low-confidence blocks** — blocks below the review threshold most likely to be mislabelled.

> Confidence is only available for runs executed after this feature was added.
> If the output files predate it, all blocks default to `confidence = 1.0`.

In [ ]:
from dmpbridge.evaluation.evaluate import load_confidence, confidence_calibration_df

REVIEW_THRESHOLD = 0.75

conf_df = load_confidence(f"{MODEL_NAME}_whole_doc")

if conf_df.empty:
    print("No confidence data found — run the experiment first.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    for lbl, grp in conf_df.groupby("label"):
        ax.hist(grp["confidence"], bins=20, range=(0, 1),
                alpha=0.55, label=lbl, histtype="stepfilled", linewidth=0.8)
    ax.axvline(REVIEW_THRESHOLD, color="#dc2626", linestyle="--",
               linewidth=1.3, label=f"Review threshold ({REVIEW_THRESHOLD})")
    ax.set_xlabel("Confidence score"); ax.set_ylabel("Block count")
    ax.set_title(f"Whole-doc — {MODEL_NAME}", pad=8)
    ax.legend(fontsize=8, framealpha=0.9); sns.despine(ax=ax)
    plt.suptitle(f"Confidence distribution per label — {MODEL_NAME}", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    cal = confidence_calibration_df(conf_df)
    bars = ax.bar(range(len(cal)), cal["accuracy"] * 100,
                  color=COLOR, alpha=0.8, edgecolor="white", width=0.6)
    for bar, row in zip(bars, cal.itertuples()):
        if row.count > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                    f"{row.accuracy*100:.0f}%\n(n={row.count})",
                    ha="center", va="bottom", fontsize=8)
    ax.plot(range(len(cal)), cal["mid"] * 100, "k--", linewidth=1.2,
            label="Ideal calibration", zorder=5)
    ax.set_xticks(range(len(cal)))
    ax.set_xticklabels(cal["bucket_label"], rotation=25, ha="right")
    ax.set_ylim(0, 115); ax.set_ylabel("Actual accuracy (%)")
    ax.set_title(f"Whole-doc — accuracy per confidence bucket", pad=8)
    ax.legend(fontsize=9); sns.despine(ax=ax)
    plt.suptitle(f"Confidence calibration — {MODEL_NAME}", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

    low = conf_df[conf_df["confidence"] < REVIEW_THRESHOLD].sort_values("confidence")
    print(f"\nLow-confidence blocks (confidence < {REVIEW_THRESHOLD}): {len(low)}")
    if not low.empty:
        pd.set_option("display.max_colwidth", 90)
        display(low[["sample", "page", "confidence", "label", "gold_label", "correct", "text"]]
                .reset_index(drop=True))

## 5 — Mislabeled blocks

Use this table to distinguish **systematic** errors (same true→pred pair, many blocks) from **isolated** ones.  
Systematic patterns → prompt or strategy needs fixing. Isolated → acceptable noise.

In [ ]:
def error_breakdown(df_err, label):
    print(f"  {label}: {len(df_err)} mislabeled blocks")
    if df_err.empty:
        print("  No errors.\n"); return
    bd = (df_err.groupby(["true","pred"]).size()
          .reset_index(name="count").sort_values("count", ascending=False))
    print(bd.to_string(index=False)); print()

error_breakdown(errs, f"Whole-doc — {MODEL_NAME}")

In [ ]:
pd.set_option("display.max_colwidth", 100)
if not errs.empty:
    display(errs[["sample","page","true","pred","text"]])